## IMPORT LIBRARY AND PATH CONFIGURATION

In [1]:
import sys
from pathlib import Path
import numpy as np

# Add python/ to PYTHONPATH (project root for Python code)
PROJECT_ROOT = Path(".").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Python project root:", PROJECT_ROOT)


Python project root: C:\UNIFI\LM Bio UniFi\Thesis\VIRTUAL-ENVIRONMENT-DEVELOPMENT-FOR-KNEE-JOINT-MOVEMENT-SIMULATION\Python


## IMPORT MODULES

In [2]:
from scripts.data_extraction_from_motive import Read_dataRB
from scripts.VirtualPoints_coordinate_extraction import export_virtual_points
from scripts.MotiveWorld_coordinate import Global_points
from scripts.UnityWorld_coordinate import Mesh_to_World_Pipeline
from scripts.Pipeline_Validation  import load_mesh_to_world, knee_flexion_angle, RMS_error 

### **READ RAW DATA FROM MOTIVE**

In [3]:
bodies = {
    "Tibia": "tibiaTransform.csv",
    "Femur": "femurTransform.csv",
     "Patella": "PatellaTransform.csv",
}

Read_dataRB(
    src_csv="../data/csv_files/dataRigidBody.csv",
    output_dir="../data/csv_files",
    bodies=bodies
)


File created: ../data/csv_files/tibiaTransform.csv
File created: ../data/csv_files/femurTransform.csv
File created: ../data/csv_files/PatellaTransform.csv


### **LOAD VP COORDIANTE ACQUIRED IN MATLAB**

In [4]:
virtual_points = {
    "Femur": {
        "points": ["femurAFlocal", "femurLFlocal", "femurMFlocal"],
        "output": "femurVP.csv",
    },
    "Tibia": {
        "points": ["tibiaALTlocal", "tibiaAMTlocal", "tibiaLTlocal", "tibiaMTlocal"],
        "output": "tibiaVP.csv",
    },
    "Patella": {
        "points": ["patellaDPlocal", "patellaMPlocal", "patellaLPlocal", "patellaPPlocal"],
        "output": "patellaVP.csv"
    }
    }

export_virtual_points(mat_file="../data/matlab/virtualPoint_DEMO.mat",
                      output_dir="../data/csv_files",
                      virtual_points=virtual_points )

File created: ..\data\csv_files\femurVP.csv
File created: ..\data\csv_files\tibiaVP.csv
File created: ..\data\csv_files\patellaVP.csv


c:\UNIFI\LM Bio UniFi\Thesis\VIRTUAL-ENVIRONMENT-DEVELOPMENT-FOR-KNEE-JOINT-MOVEMENT-SIMULATION\venv_knee_project\Lib\site-packages\scipy\io\matlab\_mio.py:235: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Considerscipy.io.matlab.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


### **COMPUTATION OF MOTIVE GLOBAL COORDINATES USING q(w,x,y,z) & T(x,y,z) PROVIDED BY MOTIVE**

In [5]:
Global_points(tibia_transform_csv="../data/csv_files/tibiaTransform.csv",
              femur_transform_csv="../data/csv_files/femurTransform.csv",
              patella_transform_csv="../data/csv_files/patellaTransform.csv",
              femur_vp_csv="../data/csv_files/femurVP.csv",
              tibia_vp_csv="../data/csv_files/tibiaVP.csv",
              patella_vp_csv="../data/csv_files/patellaVP.csv",
              output_dir="../data/csv_files",
              output_RT_dir="../data/Rigid_Transformation")

Global points generated successfully.


### **COMPUTATION OF R,T FROM MESH LOCAL REFERENCE FRAME TO MOTIVE GLOBAL WORLD AND COMPUTATION OF UNITY CSV FOR ANIMATION**

In [ ]:
Mesh_to_World_Pipeline(
              tibia_vp_csv="../data/csv_files/tibiaVP.csv",
              femur_vp_csv="../data/csv_files/femurVP.csv",
              tibia_transform_csv="../data/csv_files/tibiaTransform.csv",
              femur_transform_csv="../data/csv_files/femurTransform.csv",
              output_dir="../data/csv_files",
              output_RT_dir="../data/Rigid_Transformation")

### **METRICS FOR VALIDARION AND GRAPHICS**

In [ ]:
meshtibia_local_unity = np.array([
        [-0.02015125,  0.4032193,   0.03225459],  # AMT
        [ 0.03259825,  0.01721638, -0.01066572],  # LT
        [-0.03612585,  0.02022061,  0.01776088],  # MT
    ], dtype=float).T

meshfemur_local_unity = np.array([
        [-0.002581612, -0.4347238,   0.04451473],   # AF
        [ 0.03755924,  -0.001844588, -0.01695582],  # LF
        [-0.03962225,  -0.004840371,  0.01212757],  # MF
    ], dtype=float).T

C = np.array(
        [[-1, 0, 0],
         [ 0, 1, 0],
         [ 0, 0,-1]],
        dtype=float
    )

meshtibia_local = C @ meshtibia_local_unity
meshfemur_local = C @ meshfemur_local_unity

R_tibia,T_tibia=load_mesh_to_world("../data/Rigid_Transformation/mesh_to_world_tibia.npz")
R_femur,T_femur=load_mesh_to_world("../data/Rigid_Transformation/mesh_to_world_femur.npz")

#Flexion angle
angle=knee_flexion_angle(R_tibia,T_tibia,R_femur,T_femur,plot=True)
delta_angle=np.max(angle)-np.min(angle)
print(f"starting angle: {np.min(angle)}°")
print(f"ending angle: {np.max(angle)}°\n")
print(f"angle variation: {delta_angle:.2f}°\n")

#RMS Error
errore_tibia,errore_femur=RMS_error(R_tibia,T_tibia,R_femur,T_femur,meshtibia_local, meshfemur_local,
                                    "../data/csv_files/tibiaGlobalPoints.csv",
                                    "../data/csv_files/femurGlobalPoints.csv")
print("Tibia mean RMS (mm):", errore_tibia.mean()*1000)
print("Femur mean RMS (mm):", errore_femur.mean()*1000)